# Training hyperparameter optimization: PyTorch & Optuna

**Optuna Dashboard:** To monitor optimization progress in real-time, start the Optuna dashboard in a terminal:

```bash
cd /workspaces/CIFAR10
optuna-dashboard sqlite:///data/pytorch/training_optimization.db
```

Then open http://localhost:8080 in your browser.

## 1. Notebook setup

### 1.1. Imports

In [ ]:
# Standard library imports
import pickle

# Third party imports
import matplotlib.pyplot as plt
import numpy as np
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets

# Package imports
from image_classification_tools.pytorch import DataPipeline
import image_classification_tools.pytorch.evaluation as eval_utils
import image_classification_tools.pytorch.hyperparameter_optimization as optimization
import image_classification_tools.pytorch.plotting as plots
import image_classification_tools.pytorch.training as training

# Local imports
import configuration as config
import helper_functions as hf

### 1.2. Run configuration

In [ ]:
# Model control
rerun_optuna_study = True    # Set to False to load existing Optuna study from disk
retrain_model = True         # Set to False to load existing final model
model_source = 'huggingface' # 'local' or 'huggingface'

# Optuna study configuration
study_name = 'cnn_training_optimization'
study_storage = 'sqlite:///../data/pytorch/training_optimization.db'

# Parallel GPU configuration
n_parallel_workers = torch.cuda.device_count() if torch.cuda.is_available() else 1

### 1.3. Fixed hyperparameters

In [ ]:
# Data loading
preload_device = 'gpu' if torch.cuda.is_available() else 'cpu'

# Optuna optimization settings
n_trials = 100
epochs_per_trial = 50
pruner_warmup_steps = 5
trial_early_stopping_patience = 10

# Final model training settings
epochs = 500
early_stopping_patience = 15
print_every = 20

## 2. Load best architecture from architecture optimization

We'll load the best architecture hyperparameters from notebook 04.

In [ ]:
# Load architecture optimization study from notebook 04
architecture_study = optuna.load_study(
    study_name='cnn_architecture_optimization',
    storage='sqlite:///../data/pytorch/cnn_optimization.db'
)

# Get best architecture parameters
arch_params = architecture_study.best_trial.params

print('Best architecture hyperparameters from notebook 04:\n')
for key, value in arch_params.items():
    print(f'  {key}: {value}')

print(f'\nBest validation accuracy: {architecture_study.best_value:.2f}%')